In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# Kaggle Notebook: RNN(LSTM) for T(degC) with 12-step history
# - Train/Val on chronological data
# - Inference on lagged test (columns like "feat [t-k]")
# - Saves submission to /kaggle/working/submission.csv
# ============================================================

# -----------------------------
# 0) Imports & Config
# -----------------------------
import os, re, sys, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

SEED       = 42
SEQ_LEN    = 12        
BATCH_SIZE = 256
EPOCHS     = 15
LR         = 1e-3
HIDDEN     = 64
LAYERS     = 2
DROPOUT    = 0.1

TRAINVAL_CSV = "/kaggle/input/csu-2025-jena-climate-temperature-estimation/train_val.csv"
TEST_CSV     = "/kaggle/input/csu-2025-jena-climate-temperature-estimation/test.csv"

CKPT_PATH    = "/kaggle/working/rnn_temp.pth"
SUBMIT_PATH  = "/kaggle/working/submission.csv"

# Reproducibility
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# 1) Utils: column & sequence
# -----------------------------
def find_datetime_col(df: pd.DataFrame):
    # Prefer columns that include both "date" and "time"
    cands = [c for c in df.columns if ("date" in c.lower() and "time" in c.lower())]
    priors = ["date time", "date_time", "datetime", "time", "Date Time"]
    for c in priors:
        if c in df.columns:
            return c
    return cands[0] if cands else None

def find_target_col(df: pd.DataFrame):
    priors = ["T (degC)", "T(degC)", "T_degC", "T", "temp", "temperature"]
    for c in priors:
        if c in df.columns:
            return c
    for c in df.columns:
        if "degc" in c.lower():
            return c
    return None

def make_sequences(X_scaled: np.ndarray, y_scaled: np.ndarray, L: int):
    Xs, ys = [], []
    for i in range(L, len(X_scaled)):
        Xs.append(X_scaled[i-L:i])   # (L, F)
        ys.append(y_scaled[i])       # scalar
    return np.stack(Xs).astype(np.float32), np.array(ys, dtype=np.float32)

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]



In [ ]:
# -----------------------------
# 2) Model
# -----------------------------
class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout
        )
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        # x: (B, L, F)
        out, _ = self.lstm(x)     # (B, L, H)
        out = out[:, -1, :]       # last timestep
        out = self.head(out)      # (B, 1)
        return out.squeeze(-1)    # (B,)



In [ ]:
# -----------------------------
# 3) Load Train/Val, Prep, Train
# -----------------------------
print("Loading train/val:", TRAINVAL_CSV)
df = pd.read_csv(TRAINVAL_CSV, low_memory=False)

# datetime
dt_col = find_datetime_col(df)
assert dt_col is not None, f"Datetime column not found. Columns: {list(df.columns)}"
df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")
df = df.dropna(subset=[dt_col]).sort_values(dt_col).reset_index(drop=True)

# target
target_col = find_target_col(df)
assert target_col is not None, f"Target column (T degC) not found. Columns: {list(df.columns)}"
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

# features (numeric only, excluding target)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c != target_col]
assert len(feature_cols) > 0, "No numeric features found."

# time-ordered split 80/20
n = len(df)
split_ix = int(n * 0.8)

X_train_raw = df.iloc[:split_ix][feature_cols].values.astype(np.float32)
y_train_raw = df.iloc[:split_ix][target_col].values.astype(np.float32)
X_val_raw   = df.iloc[split_ix:][feature_cols].values.astype(np.float32)
y_val_raw   = df.iloc[split_ix:][target_col].values.astype(np.float32)

# standardize (fit on train)
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train  = x_scaler.fit_transform(X_train_raw)
X_val    = x_scaler.transform(X_val_raw)
y_train  = y_scaler.fit_transform(y_train_raw.reshape(-1,1)).reshape(-1)
y_val    = y_scaler.transform(y_val_raw.reshape(-1,1)).reshape(-1)

# sequences
X_tr, y_tr = make_sequences(X_train, y_train, SEQ_LEN)
X_va, y_va = make_sequences(X_val,   y_val,   SEQ_LEN)

train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SeqDataset(X_va, y_va), batch_size=BATCH_SIZE*2, shuffle=False)

print(f"[Data] train windows: {len(X_tr)}, val windows: {len(X_va)}, features: {len(feature_cols)}")
print(f"[Cols] datetime: {dt_col}, target: {target_col}")

# model & train
model = LSTMRegressor(n_features=len(feature_cols)).to(device)
crit  = nn.MSELoss()
opt   = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")
for ep in range(1, EPOCHS+1):
    model.train()
    tr_sum, tr_n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = crit(pred, yb)
        loss.backward()
        opt.step()
        tr_sum += loss.item() * len(xb); tr_n += len(xb)
    tr_mse = tr_sum / max(1, tr_n)

    model.eval()
    va_sum, va_n = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = crit(pred, yb)
            va_sum += loss.item() * len(xb); va_n += len(xb)
    va_mse = va_sum / max(1, va_n)

    if va_mse < best_val:
        best_val = va_mse
        torch.save({
            "model_state_dict": model.state_dict(),
            "feature_cols": feature_cols,
            "seq_len": SEQ_LEN,
            "x_scaler": x_scaler,
            "y_scaler": y_scaler,
            "target_col": target_col,
        }, CKPT_PATH)

    print(f"Epoch {ep:02d}/{EPOCHS} | train MSE: {tr_mse:.6f}  val MSE: {va_mse:.6f}")



In [ ]:
# load best & report denormalized metrics
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
x_scaler = ckpt["x_scaler"]; y_scaler = ckpt["y_scaler"]

model.eval()
val_preds_s, val_truth_s = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        val_preds_s.append(pred); val_truth_s.append(yb.numpy())

val_preds_s = np.concatenate(val_preds_s, axis=0)
val_truth_s = np.concatenate(val_truth_s, axis=0)
val_preds   = y_scaler.inverse_transform(val_preds_s.reshape(-1,1)).reshape(-1)
val_truth   = y_scaler.inverse_transform(val_truth_s.reshape(-1,1)).reshape(-1)

RMSE = float(np.sqrt(np.mean((val_preds - val_truth)**2)))
MAE  = float(np.mean(np.abs(val_preds - val_truth)))
print(f"[Validation] RMSE: {RMSE:.4f} degC, MAE: {MAE:.4f} degC")

# (Optional) quick plot inside Kaggle
try:
    n_show = min(300, len(val_preds))
    plt.figure(figsize=(10,4))
    plt.plot(val_truth[:n_show], label="True")
    plt.plot(val_preds[:n_show], label="Pred")
    plt.title("Validation: True vs Predicted T (degC)")
    plt.xlabel("Validation sample index (time-ordered)")
    plt.ylabel("T (degC)")
    plt.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print("Plot skipped:", e)


In [ ]:
# -----------------------------
# 4) Inference on lagged test (SIMPLE VERSION)
# -----------------------------
print("Loading test:", TEST_CSV)
df_test = pd.read_csv(TEST_CSV, low_memory=False)

id_col = None
for cand in ["ID", "Id", "id", "sample_id", "SampleID"]:
    if cand in df_test.columns:
        id_col = cand; break
if id_col is None:
    df_test = df_test.copy()
    df_test.insert(0, "ID", np.arange(1, len(df_test)+1))
    id_col = "ID"

assert SEQ_LEN == 12, "이 단순 버전은 SEQ_LEN=12(=t-11..t-1)만 가정합니다."
lags_to_use = list(range(11, 0, -1))

expected_cols = []
for k in lags_to_use:
    for f in feature_cols:
        expected_cols.append(f"[t-{k}] {f}")

missing_cols = [c for c in expected_cols if c not in df_test.columns]
if missing_cols:
    print("Some expected columns are missing, e.g.:", missing_cols[:10])
    raise KeyError(f"{len(missing_cols)} test columns missing. "
                   f"테스트 파일의 열 이름이 '{feature_cols[0]} [t-11]' 같은 패턴인지 확인하세요.")

try:
    feat_means = np.asarray(x_scaler.mean_).reshape(1, -1)  # (1, F)
except Exception:
    feat_means = np.zeros((1, len(feature_cols)), dtype=np.float32)

pred_rows = []

for idx, row in df_test.iterrows():
    seq = np.zeros((len(lags_to_use), len(feature_cols)), dtype=np.float32)

    for t_i, k in enumerate(lags_to_use):
        feat_vec = np.empty((1, len(feature_cols)), dtype=np.float32)
        for j, f in enumerate(feature_cols):
            colname = f"[t-{k}] {f}"
            val = row[colname]
            try:
                val = float(val)
            except Exception:
                val = np.nan
            feat_vec[0, j] = val

        if np.isnan(feat_vec).any():
            feat_vec = np.where(np.isnan(feat_vec), feat_means, feat_vec).astype(np.float32)

        try:
            feat_vec_s = x_scaler.transform(feat_vec)  # (1,F)
        except Exception:
            feat_vec_s = feat_vec
        seq[t_i, :] = feat_vec_s.squeeze(0)

    with torch.no_grad():
        inp = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 12, F)
        pred_s = model(inp).cpu().numpy().squeeze()

    try:
        pred_degC = float(y_scaler.inverse_transform(np.array(pred_s).reshape(-1,1)).reshape(-1)[0])
    except Exception:
        pred_degC = float(pred_s)

    pred_rows.append({id_col: row[id_col], "pred_T_degC": pred_degC})

# 저장
submission = pd.DataFrame(pred_rows)
submission.to_csv(SUBMIT_PATH, index=False)
print(f"Saved submission -> {SUBMIT_PATH}  (rows: {len(submission)})")
print(submission.head())

In [ ]:
df_test